In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("tipo", "csv")
dbutils.widgets.text("file", "renovacion_prestamo_clientes_info_operacional")
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_project_ssm")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsinhueproject")

In [0]:
tipo = dbutils.widgets.get("tipo")
file = dbutils.widgets.get("file")
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/{file}.{tipo}"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
file_info_operacional_schema = StructType(fields=[
    StructField("ID", IntegerType(), False),
    StructField("CLIENTE", IntegerType(), False),
    StructField("MES", IntegerType(), True),
    StructField("EDAD", IntegerType(), True),
    StructField("SEXO", StringType(), True),
    StructField("EST_CIVIL", StringType(), True),
    StructField("REGION", StringType(), True),
    StructField("FLAG_LIMA_PROVINCIA", IntegerType(), True),
    StructField("ANTIGUEDAD_MES", IntegerType(), True),
    StructField("RESENCIA_OFERTA_PLD_RENOVADO", IntegerType(), True),
    StructField("FLAG_VENTA", IntegerType(), True)
])

In [0]:
df_info_operacional = spark.read\
.option("sep", ";") \
.option('header', True)\
.schema(file_info_operacional_schema)\
.csv(ruta)

In [0]:
df_renamed_info_operacional = df_info_operacional\
    .withColumnRenamed("FLAG_LIMA_PROVINCIA", "Flag_LimProv")\
    .withColumnRenamed("RESENCIA_OFERTA_PLD_RENOVADO", "Meses_oferta")

In [0]:
df_renamed_info_operacional.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.{file}")